# Customer retention

### Who returns within 90 days of their first observed purchase?

Python preparation → Snowflake features → prediction → usefulness

## 1 · Start with the raw records

UCI Online Retail · 541,909 product rows · 8 fields

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)


In [ ]:
raw_data = pd.read_excel(ROOT / "data/raw/Online Retail.xlsx")


In [ ]:
raw_data.head()

In [ ]:
raw_data.shape

## 2 · Make customers traceable

CustomerID present · 406,829 rows

In [ ]:
raw_data.isna().sum()

In [ ]:
customer_data = raw_data.dropna(subset=["CustomerID"])

In [ ]:
customer_data.shape

## 3 · Keep valid purchase lines

Positive quantity · Positive unit price · 397,884 rows

In [ ]:
purchase_data = customer_data[
    (customer_data["Quantity"] > 0)
    & (customer_data["UnitPrice"] > 0)
]

purchase_data[["Quantity", "UnitPrice"]].describe()

## 4 · Hand prepared data to Snowflake

In [ ]:
purchase_data.to_csv(ROOT / "data/processed/purchases.csv", index=False)


### Next · Build customer features, test a prediction, evaluate usefulness

## Reference: data checks and the 90-day outcome
Original preparation calculations, with saved outputs cleared for a clean rerun.


### Inspect excluded quantities and prices

In [ ]:
customer_data[["Quantity", "UnitPrice"]].describe()

In [ ]:
customer_data[customer_data["Quantity"] < 0].head()

### One invoice, one order date

Customer + invoice · Earliest invoice timestamp

Repeated invoice entries: 30 before aggregation → 0 after

In [ ]:
orders = purchase_data[["CustomerID", "InvoiceNo", "InvoiceDate"]].drop_duplicates()

orders.head()

In [ ]:
orders["InvoiceNo"].duplicated().sum()

In [ ]:
(
    orders[orders["InvoiceNo"].duplicated(keep=False)]
    .sort_values("InvoiceNo")
    .head(10)
)

In [ ]:
orders.groupby("InvoiceNo")["CustomerID"].nunique().max()

In [ ]:
orders = (
    purchase_data.groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"]
    .min()
    .reset_index()
)

orders.head()

In [ ]:
orders["InvoiceNo"].duplicated().sum()

### Any repeat purchase across the dataset

Exploratory measure · Different from the 90-day outcome

In [ ]:
order_counts = orders.groupby("CustomerID").size()

order_counts.head()

In [ ]:
repeated_purchases = order_counts[order_counts > 1]

repeated_purchases.head()

In [ ]:
len(repeated_purchases)

In [ ]:
len(repeated_purchases) / len(order_counts) * 100

### Give every customer the full 90 days

First observed purchase ≤ 10 September 2011, 12:50 · 3,370 eligible customers

First observed ≠ first-ever purchase

In [ ]:
print(orders["InvoiceDate"].min())
print(orders["InvoiceDate"].max())

In [ ]:
cutoff_date = orders["InvoiceDate"].max() - pd.Timedelta(days=90)

cutoff_date

In [ ]:
first_purchase = orders.groupby("CustomerID")["InvoiceDate"].min()
first_purchase.head()

In [ ]:
eligible_customers = first_purchase[first_purchase <= cutoff_date]
eligible_customers.head()

In [ ]:
len(eligible_customers)

### Identify the second distinct order

In [ ]:
sorted_orders = orders.sort_values(["CustomerID", "InvoiceDate"])

sorted_orders.head()

In [ ]:
sorted_orders["order_number"] = sorted_orders.groupby("CustomerID").cumcount() + 1
sorted_orders.head()

In [ ]:
second_orders = sorted_orders[sorted_orders["order_number"] == 2]
second_orders.head()

In [ ]:
second_purchase = second_orders.set_index("CustomerID")["InvoiceDate"]

time_to_second = second_purchase - first_purchase

time_to_second.head()

### Expected historical result: 47.63% returned within 90 days
1,605 of 3,370 eligible customers. Rerun below to verify.


In [ ]:
eligible_times = time_to_second.loc[eligible_customers.index]

returned_within_90 = eligible_times <= pd.Timedelta(days=90)

returned_within_90.mean() * 100

In [ ]:
returned_within_90.sum()